In [29]:
from bs4 import BeautifulSoup as bs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
import pandas as pd

In [ ]:
def scrap_unis_base(url, browser):
    '''
    Function made to scrap the base info of spanish universities
    
    Args: 
        url (str): url of the webpage to scrape
        browser: initialized browser for webscrapping with uc
    
    Returns: 
        universities (list of dicts): raw data of universities
    '''
    
    browser.get(url)
    browser.implicitly_wait(0.5)
    
    html = browser.page_source
    soup = bs(html, "html.parser")
    
    universities = []

    # Recorremos cada comunidad
    for comunidad_div in soup.find_all("div", class_="divComunidad"):
        # Para cada universidad dentro de la comunidad
        for uni_row in comunidad_div.find_all("tr", class_="rowComunidad"):
            name_td = uni_row.find("td", class_="col1")
            province_td = uni_row.find("td", class_="col3")
            website_td = uni_row.find("td", class_="col4")
            type_td = uni_row.find("td", class_="col2")

            name = name_td.get_text(strip=True) if name_td else ""
            province = province_td.get_text(strip=True) if province_td else ""
            website = website_td.find("a")["href"] if website_td and website_td.find("a") else ""
            uni_type = type_td.get_text(strip=True) if type_td else ""
            
            universities.append({
                "name": name,
                "website": website,
                "province": province,
                "type": uni_type
            })
            
    return universities

In [48]:
import pandas as pd
df = pd.read_json("/Users/pablotutorlopez/Desktop/CARPETAS/Match_Key_Mavericks/MatchKey-GenAI-Maverick/backend/app/services/scraping/universities/data/universities_raw.json", orient="records")
df.head()

,name,website,province,type
0,Universidad de Almería,https://www.ual.es,Almería,Pública
1,Universidad de Cádiz,https://www.uca.es,Cádiz,Pública
2,Universidad de Córdoba,http://www.uco.es,Córdoba,Pública
3,Universidad Loyola Andalucía,https://www.uloyola.es,Córdoba,Privada
4,Universidad de Granada,https://www.ugr.es,Granada,Pública


In [79]:
urls = list(df["website"].unique())

## 19 nov

In [283]:
import time
from selenium.webdriver.common.action_chains import ActionChains

PRIORITY_GROUPS = [
    ["identidad-y-mision"],
    ["valores"],
    ["saludo-rector"],
    ["historia"],
    ["mision", "misión", "vision", "visión"],
    ["quienes-somos", "nosotros", "quienes", "about"],
    ["planestrategico", "plan-estrategico", "plan_estrategico"],
]


def open_menus(browser):
    """Intenta abrir menús desplegables moviendo el ratón por posibles elementos del menú"""
    try:
        menu_elements = browser.find_elements(By.CSS_SELECTOR, "nav, li, dropdown, menu")
        for el in menu_elements:
            try:
                ActionChains(browser).move_to_element(el).perform()
            except:
                pass
        time.sleep(0.2)  # dejar un momento para que se carguen los submenús
    except:
        pass


def get_relevant_links(browser, urls):
    
    universities = []
    
    for url in urls:

        links = []
        
        browser.get(url)
        browser.implicitly_wait(2)
        
        open_menus(browser)
        
        html = browser.page_source
        soup = bs(html, "html.parser")
        
        for a in soup.find_all("a", href=True):
            full_link = urljoin(url, a["href"])
            links.append(full_link)
            
        chosen_link = None
            
        for group in PRIORITY_GROUPS:
            matches = [l for l in links if any(k in l.lower() for k in group)]
            if matches:
                chosen_link = matches[0]       # coger el primero válido
                break   
        
        universities.append(chosen_link)    
        time.sleep(0.1)
        
            
    return universities
        
# options = uc.ChromeOptions()
# options.add_argument("--headless=chrome")
# options.add_argument("--disable-gpu")
# options.add_argument("--disable-extensions")
# options.add_argument("--disable-logging")
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")
browser = uc.Chrome()
universities_links = get_relevant_links(browser=browser, urls=urls[:2])
browser.quit()

universities_links

['https://www.ual.es/planestrategico', 'https://www.uca.es#valoresUCA']

In [ ]:
import time
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException, TimeoutException
from urllib.parse import urljoin
from bs4 import BeautifulSoup as bs
import undetected_chromedriver as uc

PRIORITY_GROUPS = [
    ["/portal/impe/web/portadaEspecifica?channel="],
    ["identidad-y-mision"],
    ["valores"],
    ["saludo-rector"],
    ["historia"],
    ["mision", "misión", "vision", "visión", "misiones"],
    ["quienes-somos", "nosotros", "quienes", "about"],
    ["planestrategico", "plan-estrategico", "plan_estrategico", "Plan-Estrategico"],
]

def open_main_menu(browser):
    """Hace hover sobre los elementos principales del menú para desplegar submenús"""
    try:
        nav_elements = browser.find_elements(By.CSS_SELECTOR, "nav")
        for nav in nav_elements:
            links = nav.find_elements(By.TAG_NAME, "a")
            for link in links:
                try:
                    ActionChains(browser).move_to_element(link).perform()
                except:
                    continue
        time.sleep(0.2)
    except:
        pass

def extract_links_from_elements(elements, base_url):
    links = set()
    for el in elements:
        try:
            href = el.get_attribute("href")
            if href:
                links.add(urljoin(base_url, href))
        except:
            continue
    return list(links)

def get_relevant_links(browser, urls):
    universities = []

    for url in urls:
        chosen_link = None

        # Normalizar URL
        if not url.startswith("http"):
            url = "https://" + url.lstrip("http://").lstrip("https://")

        try:
            browser.set_page_load_timeout(15)
            browser.get(url)
            browser.implicitly_wait(1)
        except (WebDriverException, TimeoutException) as e:
            print(f"No se pudo cargar {url}: {e}")
            universities.append(None)
            continue

        # Buscar en menú principal
        open_main_menu(browser)
        try:
            nav_elements = browser.find_elements(By.CSS_SELECTOR, "nav a")
            links = extract_links_from_elements(nav_elements, url)
            for group in PRIORITY_GROUPS:
                matches = [l for l in links if any(k in l.lower() for k in group)]
                if matches:
                    chosen_link = matches[0]
                    break
        except:
            pass

        # Buscar en footer si no se encontró
        if not chosen_link:
            try:
                footer_elements = browser.find_elements(By.CSS_SELECTOR, "footer a")
                links = extract_links_from_elements(footer_elements, url)
                for group in PRIORITY_GROUPS:
                    matches = [l for l in links if any(k in l.lower() for k in group)]
                    if matches:
                        chosen_link = matches[0]
                        break
            except:
                pass

        # Buscar en todo el HTML como último recurso
        if not chosen_link:
            try:
                html = browser.page_source
                soup = bs(html, "html.parser")
                all_links = [urljoin(url, a["href"]) for a in soup.find_all("a", href=True)]
                for group in PRIORITY_GROUPS:
                    matches = [l for l in all_links if any(k in l.lower() for k in group)]
                    if matches:
                        chosen_link = matches[0]
                        break
            except:
                pass

        universities.append(chosen_link)
        time.sleep(0.1)  # espera mínima entre URLs

    return universities

# Ejemplo de uso:
browser = uc.Chrome()

# Suponiendo que urls es tu lista de universidades
universities_links = get_relevant_links(browser=browser, urls=urls)
browser.quit()

print(universities_links)

No se pudo cargar http://www.unica.edu: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: chrome=142.0.7444.176)
Stacktrace:
0   undetected_chromedriver             0x00000001032a5698 undetected_chromedriver + 6153880
1   undetected_chromedriver             0x000000010329cb6a undetected_chromedriver + 6118250
2   undetected_chromedriver             0x0000000102d31a5b undetected_chromedriver + 436827
3   undetected_chromedriver             0x0000000102d287a6 undetected_chromedriver + 399270
4   undetected_chromedriver             0x0000000102d198d8 undetected_chromedriver + 338136
5   undetected_chromedriver             0x0000000102d1b6fb undetected_chromedriver + 345851
6   undetected_chromedriver             0x0000000102d19e13 undetected_chromedriver + 339475
7   undetected_chromedriver             0x0000000102d19669 undetected_chromedriver + 337513
8   undetected_chromedriver             0x0000000102d19369 undetected_chromedriver + 336745
9   undetected_chromedriver

In [301]:
universities_links

['https://www.ual.es/planestrategico',
 'https://www.uca.es/historia/',
 'https://www.uco.es/historia-y-video-promocional.html',
 'https://www.uloyola.es/universidad/quienes-somos/identidad-y-mision',
 'https://www.ugr.es/universidad/organizacion/saludo-rector',
 'https://www.uhu.es/la-uhu/historia',
 'http://www.ujaen.es/la-universidad/mision-vision-y-valores-de-la-uja',
 'https://www.uma.es/historiadigital/',
 'https://www.unia.es/la-unia/historia-de-la-unia',
 None,
 'https://www.us.es/laUS/historia',
 'https://www.usj.es/conocenos/mision-vision-valores',
 'https://www.unizar.es/institucion/conoce-la-universidad/cultura-y-valores',
 'https://www.uniovi.es/conocenos/uniovi/historia',
 None,
 'https://www.ull.es/la-universidad/historia-mision-vision-valores',
 'https://web.unican.es/admision',
 'https://www.ucavila.es/universidad/saludo-rectora/',
 'https://www.ubu.es/acceso-admision-y-matricula/matricula/matricula-de-grado',
 'https://www.unileon.es/universidad/localizacion-e-histori

In [303]:
df = pd.DataFrame(universities_links, columns=["url_about_us"])

# Ruta donde guardar el CSV
path_csv = "/Users/pablotutorlopez/Desktop/CARPETAS/Match_Key_Mavericks/MatchKey-GenAI-Maverick/backend/app/services/scraping/universities/data/universities_links.csv"

# Guardarlo
df.to_csv(path_csv, index=False, encoding="utf-8")

In [260]:
import json

with open("/Users/pablotutorlopez/Desktop/CARPETAS/Match_Key_Mavericks/MatchKey-GenAI-Maverick/backend/app/services/scraping/universities/data/universities_links.json", "w", encoding="utf-8") as f:
    json.dump(universities_links, f, ensure_ascii=False, indent=4)

In [290]:
browser.quit()